# Tapping Features Data Exploration

This notebook explores tapping test data extracted from raw JSON files. It provides:
1. Overview of the paired dataset (PD patients and healthy controls)
2. Data loading and basic statistics
3. Comparison of train/val/test splits
4. Exploration of session-level tapping features
5. Analysis of advanced patient-level and session-level features

**Key Outputs:**
- Paired healthcodes: Demographics and diagnosis labels
- Train/Val/Test splits: Cross-validation structure
- Tapping features: Statistical and advanced metrics
- Feature distributions by diagnosis class

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# ===== Load Paired Dataset =====
# Paired healthcodes with diagnosis labels (PD patients and matched healthy controls)
paired = pd.read_csv("/mloscratch/users/clerget/NeuroMeditron/src_GAMMA/paired_healthcode.csv")

# ===== Load Cross-Validation Splits =====
# 5-fold CV splits for training (used in this project: 5-fold)
train_split = pd.read_csv("/mloscratch/users/clerget/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_train.csv")
valtest_split = pd.read_csv("/mloscratch/users/clerget/data/data_paired/5_fold_CV/processed_paired/paired_splits/balanced_train/healthcode_5fold_val_test.csv")

In [ ]:
print("="*80)
print("PAIRED DATASET OVERVIEW")
print("="*80)
print(f"Shape: {paired.shape}")
print(f"Columns: {list(paired.columns)}")
print("\nFirst 5 rows:")
print(paired.head())

paired dataset shape: (532, 3)


,healthCode,label_PD,modality
0,00081bd9-9abd-4003-b035-de6cc3e8c922,0,paired
1,0160664a-f4af-4071-a4aa-2967f3ea0503,1,paired
2,01aea090-d2cd-409e-8ce8-22887ada7af8,1,paired
3,01b1ec31-0348-4148-a641-626d4391b4fb,0,paired
4,01fa5bda-f11e-47ef-94c6-37697ad26a86,1,paired


In [ ]:
print("="*80)
print("TRAINING SPLIT OVERVIEW")
print("="*80)
print(f"Shape: {train_split.shape}")
print(f"Columns: {list(train_split.columns)}")
print("\nFold distribution:")
print(train_split['fold_iteration'].value_counts().sort_index())
print("\nFirst 5 rows:")
print(train_split.head())

train split shape: (3680, 3)


,fold_iteration,healthCode,subset
0,0,00081bd9-9abd-4003-b035-de6cc3e8c922,train
1,0,0160664a-f4af-4071-a4aa-2967f3ea0503,train
2,0,01aea090-d2cd-409e-8ce8-22887ada7af8,train
3,0,01b1ec31-0348-4148-a641-626d4391b4fb,train
4,0,0309d5a8-79cd-460a-9a87-6902460ab71d,train


In [ ]:
print("="*80)
print("VALIDATION/TEST SPLIT OVERVIEW")
print("="*80)
print(f"Shape: {valtest_split.shape}")
print(f"Columns: {list(valtest_split.columns)}")
print("\nSubset distribution:")
print(valtest_split['subset'].value_counts())
print("\nFold distribution:")
print(valtest_split['fold_iteration'].value_counts().sort_index())
print("\nFirst 5 rows:")
print(valtest_split.head())

val/test split shape: (1064, 3)


,fold_iteration,healthCode,subset
0,0,01fa5bda-f11e-47ef-94c6-37697ad26a86,val
1,0,030fe814-de84-4fb5-8df6-84dd6e7cc7dc,val
2,0,05d4e71a-c792-4549-8ec7-6a7dc355a1bd,val
3,0,09670b1f-23a3-46a5-8daf-f78f577b7697,val
4,0,13627002-073b-4366-8889-9561789c51be,val


In [ ]:
print("="*80)
print("HEALTHCODE CONSISTENCY ACROSS DATASETS")
print("="*80)

# Compare healthCodes across datasets
demo_codes = set(paired["healthCode"].unique())
train_codes = set(train_split["healthCode"].unique())
valtest_codes = set(valtest_split["healthCode"].unique())
all_split_codes = train_codes | valtest_codes

print(f"Paired dataset: {len(demo_codes):,} unique healthCodes")
print(f"Train split: {len(train_codes):,} unique healthCodes")
print(f"Val/Test split: {len(valtest_codes):,} unique healthCodes")
print(f"Total in splits: {len(all_split_codes):,} unique healthCodes")

# Check coverage
in_paired = len(all_split_codes & demo_codes)
missing = all_split_codes - demo_codes
print(f"\n✓ Codes in paired dataset: {in_paired:,}")
if missing:
    print(f"✗ Missing from paired: {len(missing)}")
else:
    print("✓ All split codes present in paired dataset")

Paired dataset: 532 unique healthCodes
Train split: 532 unique healthCodes
Val/Test split: 532 unique healthCodes


## Session-Level Tapping Features

Tapping test features extracted from raw JSON files. Each row represents one session/trial per patient.

### Feature Types:
- **Event Sequences**: Inter-tap intervals (Δt) and button pressed for each tap
- **Statistical Features**: Mean/std tap intervals, coordinates, missed taps, etc.

In [ ]:
print("="*80)
print("TAPPING EVENT SEQUENCES")
print("="*80)
tapping_sequences = pd.read_csv("/mloscratch/users/clerget/data/csv/tapping_event_sequences_session.csv")
print(f"Shape: {tapping_sequences.shape} (event-level, one row per tap)")
print(f"Columns: {list(tapping_sequences.columns)}")
print("\nFirst 10 rows:")
print(tapping_sequences.head(10))

Tapping event sequences shape: (3498391, 3)
                             healthCode   delta_t             button
0  5134777c-3563-41a5-abcf-cce272a2ebf3  0.429935  TappedButtonRight
1  5134777c-3563-41a5-abcf-cce272a2ebf3  0.349959   TappedButtonLeft
2  5134777c-3563-41a5-abcf-cce272a2ebf3  0.283382  TappedButtonRight
3  5134777c-3563-41a5-abcf-cce272a2ebf3  0.299909   TappedButtonLeft
4  5134777c-3563-41a5-abcf-cce272a2ebf3  0.300355   TappedButtonLeft


In [ ]:
print("="*80)
print("TAPPING STATISTICAL FEATURES (Session-level)")
print("="*80)
tapping_features = pd.read_csv("/mloscratch/users/clerget/data/csv/tapping_statistical_features_sessions.csv")
print(f"Shape: {tapping_features.shape} (one row per session/trial)")
print(f"Columns: {list(tapping_features.columns)}")
print("\nBasic statistics:")
print(tapping_features.describe())


Tapping statistical features shape: (15992, 12)
    mean_dt    std_dt  lr_ratio  left_count  right_count  total_taps  \
0  0.131566  0.115565  0.853659          70           82         152   
1  0.080679  0.081303  0.871212         115          132         247   
2  0.181918  0.114110  1.000000          54           54         108   
3  0.197994  0.122458  0.960000          48           50         101   
4  0.084800  0.071506  1.201923         125          104         236   

       mean_x      mean_y      std_x      std_y  missed_taps  \
0  143.362745  434.088235  73.065760  16.088107            0   
1  151.334677  400.203629  60.303491  14.125084            0   
2  153.674312  405.220183  78.037640  20.139813            0   
3  153.299020  437.220588  77.773624  22.610467            3   
4  178.966245  427.729958  67.726390  15.981294            7   

                             healthCode  
0  5134777c-3563-41a5-abcf-cce272a2ebf3  
1  3516da76-a3e1-47ac-9255-ee100797ba9b  
2  0a46

In [ ]:
print("="*80)
print("FEATURES MERGED WITH DIAGNOSIS LABELS")
print("="*80)
tapping_with_diagnosis = tapping_features.merge(
    paired[["healthCode", "label_PD"]], 
    on="healthCode", 
    how="left"
)

print(f"Shape: {tapping_with_diagnosis.shape}")
print(f"\nDiagnosis distribution:")
print(tapping_with_diagnosis['label_PD'].value_counts().sort_index())
print("\nFirst rows:")
print(tapping_with_diagnosis.head())

Tapping features merged with diagnosis:
    mean_dt    std_dt  lr_ratio  left_count  right_count  total_taps  \
0  0.131566  0.115565  0.853659          70           82         152   
1  0.080679  0.081303  0.871212         115          132         247   
2  0.181918  0.114110  1.000000          54           54         108   
3  0.197994  0.122458  0.960000          48           50         101   
4  0.084800  0.071506  1.201923         125          104         236   

       mean_x      mean_y      std_x      std_y  missed_taps  \
0  143.362745  434.088235  73.065760  16.088107            0   
1  151.334677  400.203629  60.303491  14.125084            0   
2  153.674312  405.220183  78.037640  20.139813            0   
3  153.299020  437.220588  77.773624  22.610467            3   
4  178.966245  427.729958  67.726390  15.981294            7   

                             healthCode  label_PD  
0  5134777c-3563-41a5-abcf-cce272a2ebf3       1.0  
1  3516da76-a3e1-47ac-9255-ee100797ba9

In [ ]:
print("="*80)
print("FEATURE STATISTICS BY DIAGNOSIS")
print("="*80)

print("\n--- Mean Δt (Inter-tap Interval) ---")
print("Label 0 = Healthy, Label 1 = PD")
print(tapping_with_diagnosis.groupby("label_PD")["mean_dt"].describe())

print("\n--- Standard Deviation Δt ---")
print(tapping_with_diagnosis.groupby("label_PD")["std_dt"].describe())

print("\n--- Total Taps per Session ---")
print(tapping_with_diagnosis.groupby("label_PD")["total_taps"].describe())

Mean Δt by diagnosis:
            count      mean       std       min       25%       50%       75%  \
label_PD                                                                        
0.0        3378.0  0.094790  0.048748  0.036418  0.057239  0.079994  0.118239   
1.0       12387.0  0.125359  0.078341  0.037261  0.076665  0.105511  0.149491   

               max  
label_PD            
0.0       0.455010  
1.0       0.929778  

L/R ratio by diagnosis:
            count      mean       std       min       25%       50%       75%  \
label_PD                                                                        
0.0        3378.0  1.018129  0.162252  0.171053  0.976744  1.000000  1.039152   
1.0       12387.0  1.105898  1.182022  0.000000  0.951807  1.009804  1.110000   

                max  
label_PD             
0.0        5.133333  
1.0       90.000000  


## Advanced Patient-Level Features

Features are aggregated at the patient level (mean across all sessions/trials).

### Feature Definitions:

**Fatigue Slope:**
- Measures change in performance over a session
- Mean reaction time in 1st quarter vs 4th quarter
- Formula: `(last - first) / first` (% change)
- Negative slope = fatigue (slower taps near end)

**Variability Index:**
- Coefficient of variation in tap intervals
- Formula: `std(Δt) / mean(Δt)`
- High variability indicates inconsistent tapping

**Tap Distance:**
- Euclidean distance between consecutive tap coordinates
- Reported as mean and std across session

**Fluctuation:**
- Variability in reaction time changes
- `std(diff(Δt))` - standard deviation of differences between consecutive intervals
- High fluctuation = unpredictable tapping pattern

**Note on NaN Values:**
- If a patient has only 1 trial, cannot compute standard deviations → NaN
- These features require multiple sessions per patient to compute meaningful statistics

In [ ]:
print("="*80)
print("ADVANCED TAPPING FEATURES (Patient-level)")
print("="*80)
advanced_features_patient = pd.read_csv("/mloscratch/users/clerget/data/csv/tapping_statistical_features_patient.csv")
print(f"Shape: {advanced_features_patient.shape} (one row per patient)")
print(f"Columns: {list(advanced_features_patient.columns)}")
print("\nBasic statistics:")
print(advanced_features_patient.describe())
print(f"\nMissing values:")
print(advanced_features_patient.isnull().sum())


Advanced tapping features shape: (608, 11)
                             healthCode  fatigue_slope_mean  \
0  00081bd9-9abd-4003-b035-de6cc3e8c922            0.312848   
1  005908f8-2bf2-4855-ab2d-ac3c0190b8fb            0.035746   
2  00d9a01f-08ae-4c3d-9b23-919f66fb066f            0.092835   
3  0160664a-f4af-4071-a4aa-2967f3ea0503            0.129422   
4  01aea090-d2cd-409e-8ce8-22887ada7af8           -0.122193   

   fatigue_slope_std  variability_index_mean  variability_index_std  \
0           0.320994                0.350326               0.190731   
1                NaN                0.781290                    NaN   
2           0.113944                1.203761               0.183419   
3           0.137016                0.848255               0.121591   
4           0.195725                0.562091               0.150828   

   mean_tap_distance_mean  mean_tap_distance_std  std_tap_distance_mean  \
0              149.108105              13.185257              37.272614   


**NaN** because if one trial, cannot compute std

In [ ]:
print("="*80)
print("FATIGUE SLOPE ANALYSIS")
print("="*80)

# Merge with diagnosis labels
merged = advanced_features_patient.merge(
    paired[["healthCode", "label_PD"]], 
    on="healthCode", 
    how="left"
)

print("\nFatigue Slope by Diagnosis:")
print("(Negative slope = fatigue, positive = improvement)")
print(merged.groupby("label_PD")["fatigue_slope_mean"].describe())

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
merged.boxplot(column='fatigue_slope_mean', by='label_PD', ax=ax)
ax.set_xlabel("Diagnosis (0=Healthy, 1=PD)")
ax.set_ylabel("Fatigue Slope")
ax.set_title("Fatigue Slope Distribution by Diagnosis")
plt.suptitle('')
plt.show()

Mean fatigue_slope_mean across all patients: 0.051810

Mean fatigue_slope_mean by diagnosis:
          count      mean       std       min       25%       50%       75%  \
label_PD                                                                      
0.0       230.0  0.066231  0.157428 -0.413070 -0.030536  0.074020  0.153172   
1.0       302.0  0.043469  0.209001 -0.772777 -0.065834  0.032554  0.133039   

               max  
label_PD            
0.0       0.575797  
1.0       1.067536  


## Advanced Session-Level Features

Advanced features computed per session/trial. Each row represents one session for one patient.

In [ ]:
print("="*80)
print("ADVANCED TAPPING FEATURES (Session-level)")
print("="*80)
advanced_features_session = pd.read_csv("/mloscratch/users/clerget/data/csv/tapping_combined_features_session.csv")
print(f"Shape: {advanced_features_session.shape} (one row per session/trial)")
print(f"Columns: {list(advanced_features_session.columns)}")
print("\nBasic statistics:")
print(advanced_features_session.describe())
print(f"\nSample rows:")
print(advanced_features_session.head())


Advanced tapping features shape: (15992, 6)
   fatigue_slope  variability_index  mean_tap_distance  std_tap_distance  \
0      -0.209819           0.878379         113.285935         61.770407   
1       0.085573           1.007742         106.095709         39.966436   
2       0.074332           0.627261         122.535063         64.082528   
3       0.040705           0.618496         135.062383         52.932558   
4       0.043829           0.843230         119.798322         43.549124   

   fluctuation                            healthCode  
0     0.185746  5134777c-3563-41a5-abcf-cce272a2ebf3  
1     0.142372  3516da76-a3e1-47ac-9255-ee100797ba9b  
2     0.201833  0a4662b0-9ee3-4029-af85-7e342cb87810  
3     0.215947  e787d06a-bcce-4ef0-848e-8b319fc7ce99  
4     0.126878  3516da76-a3e1-47ac-9255-ee100797ba9b  
